# 03 - Classical Preprocessing and Handcrafted Features

Demonstrates EXIF-safe RGB conversion, bounded resizing, CLAHE normalization, and HOG, uniform-LBP, HSV, colour-moment, edge-density, and mask-shape features. Raw files stay immutable and no pretrained representation is used.

In [ ]:
import csv,sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
cwd=Path.cwd().resolve(); REPO=next((p for p in [cwd,*cwd.parents] if (p/'implementation'/'src').exists()),None)
if REPO is None: raise RuntimeError('Start Jupyter inside the repository.')
sys.path.insert(0,str(REPO/'implementation'/'src'))
from ipcv_attire.dataset_policy import require_showcase_approval
from ipcv_attire.features import FeatureConfig,extract_handcrafted_features
from ipcv_attire.preprocessing import preprocess_image
DATA=REPO/'implementation'/'data'; MANIFEST=DATA/'interim'/'manifests'/'fashionpedia-images.csv'; DEMO_IMAGE_ID='13665'


## Approved input and normalization

Rendering fails closed unless the automatic display-risk flag is clear and the manual showcase status is approved. Training data is not content-filtered.

In [ ]:
with MANIFEST.open(encoding='utf-8',newline='') as handle: record=next(r for r in csv.DictReader(handle) if r['image_id']==DEMO_IMAGE_ID)
require_showcase_approval(DEMO_IMAGE_ID,DATA/'showcase-manifest.csv',display_risk=record['display_risk']=='1')
image_path=DATA/'raw'/'fashionpedia'/record['relative_image_path']; sample=preprocess_image(image_path,maximum_side=1024)
print({'split':record['final_split'],'original':sample.original_rgb.shape,'normalized':sample.normalized_rgb.shape,'warnings':sample.warnings})
fig,axes=plt.subplots(1,3,figsize=(15,5))
for axis,image,title in zip(axes,[sample.original_rgb,sample.resized_rgb,sample.normalized_rgb],['EXIF-safe RGB','Bounded resize','Luminance CLAHE']): axis.imshow(image); axis.set_title(title); axis.axis('off')
plt.tight_layout()


## Inspectable float32 features

HOG describes oriented edges, uniform LBP texture, HSV colour, colour moments channel distributions, edge density boundary activity, and shape statistics the segmented silhouette. Scalers fit on training only.

In [ ]:
mask=np.ones(sample.normalized_rgb.shape[:2],dtype=np.uint8); bundle=extract_handcrafted_features(sample.normalized_rgb,mask,FeatureConfig())
print(bundle.feature_lengths,'total=',bundle.vector.size,'dtype=',bundle.vector.dtype)
fig,axes=plt.subplots(1,3,figsize=(14,3)); axes[0].imshow(bundle.hog_image,cmap='gray'); axes[0].set_title('HOG'); axes[1].hist(bundle.lbp_image.ravel(),bins=10); axes[1].set_title('Uniform LBP'); axes[2].plot(bundle.hsv_histogram); axes[2].set_title('HSV'); plt.tight_layout()
